# Issue #86 — source-locked Train execution wrapper

Execution-only wrapper for canonical substrate creation, one deterministic stability replicate, or zero-choice Train finalization. Scientific constants live only in the pinned source.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
BRANCH = 'research/motif-qualification-sparsification'
SOURCE_SHA = '16a84b2b36f0d3584afd0a547487c4373dc22128'
TRAIN_SHA256 = 'deb82c4b4e01b90776a718c34934666b0bdde6696ca1d0149f8fe807a8ff4ba8'
DICTIONARY_SHA256 = '68154a054f712bb07692146904bcba57f10e079c7efc92723aa0bccba9f6273b'
TRAIN_MODEL_SHA256 = '77b8a41d4a7de79b2216b4b9c7ad2d19e326cac25a46ec2a7a3dcaaa1f95607a'

EXECUTION_MODE = os.environ.get('MOTIF_EXECUTION_MODE', 'REVIEW_ONLY')
STABILITY_ARM = os.environ.get('MOTIF_STABILITY_ARM', '')
STABILITY_REPLICATE_ID = os.environ.get('MOTIF_STABILITY_REPLICATE_ID', '')
TRAIN_CSV = Path('/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split/train.csv')
DICTIONARY_NPZ = Path('/kaggle/input/datasets/nuyntai/pgm-e01-v533-dictionary-crs/e01_dictionary.npz')
TRAIN_MODEL_NPZ = Path('/kaggle/input/datasets/nuyntai/pgm-crs-train-v2-artifacts/crs_train_model.npz')
SUBSTRATE_DIR = Path(os.environ.get('MOTIF_SUBSTRATE_DIR', '/kaggle/input/pgm-motif-train-substrate'))
REPLICATES_DIR = Path(os.environ.get('MOTIF_REPLICATES_DIR', '/kaggle/input/pgm-motif-stability-replicates'))
OUTPUT_DIR = Path('/kaggle/working/motif_issue86_outputs')
REPO_DIR = Path('/kaggle/working/FER2013_Graph')

print({'execution_mode': EXECUTION_MODE, 'arm': STABILITY_ARM or None, 'replicate_id': STABILITY_REPLICATE_ID or None, 'internet_required_for_clone': True, 'gpu_required': False})


In [ ]:
if REPO_DIR.exists():
    raise FileExistsError(f'refusing pre-existing repository path: {REPO_DIR}')
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
if head != SOURCE_SHA:
    raise RuntimeError(f'source lock mismatch: {head}')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError('source worktree is not clean')
print({'source_lock': 'PASS', 'head': head})


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR / 'research/pixel_relational_motif_e0')], check=True)
subprocess.run([sys.executable, '-m', 'pytest', str(REPO_DIR / 'research/pixel_relational_motif_e0/tests'), '-q'], check=True)
print('Pre-data full pytest PASS')


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

if EXECUTION_MODE == 'REVIEW_ONLY':
    print('REVIEW_ONLY complete: no FER input opened and no Train execution started')
else:
    if EXECUTION_MODE not in {'BUILD_SUBSTRATE', 'RUN_REPLICATE', 'FINALIZE_TRAIN'}:
        raise ValueError('invalid execution-only mode')
    required = [(TRAIN_MODEL_NPZ, TRAIN_MODEL_SHA256)]
    if EXECUTION_MODE in {'BUILD_SUBSTRATE', 'FINALIZE_TRAIN'}:
        required.append((TRAIN_CSV, TRAIN_SHA256))
    if EXECUTION_MODE == 'BUILD_SUBSTRATE':
        required.append((DICTIONARY_NPZ, DICTIONARY_SHA256))
    for path, expected in required:
        observed = sha256_file(path)
        if observed != expected:
            raise ValueError(f'input lock mismatch for {path}: {observed}')
        print({'input': str(path), 'sha256': observed})
    if EXECUTION_MODE in {'RUN_REPLICATE', 'FINALIZE_TRAIN'} and not (SUBSTRATE_DIR / 'motif_train_substrate_manifest.json').is_file():
        raise FileNotFoundError('canonical substrate manifest is missing')
    if EXECUTION_MODE == 'FINALIZE_TRAIN' and not REPLICATES_DIR.is_dir():
        raise FileNotFoundError('replicate artifact directory is missing')


In [ ]:
if EXECUTION_MODE != 'REVIEW_ONLY':
    if OUTPUT_DIR.exists():
        raise FileExistsError(f'refusing existing output directory: {OUTPUT_DIR}')
    OUTPUT_DIR.mkdir(parents=True)
    command = [sys.executable, '-m', 'pixel_relational_motif_e0.motif_train_runner']
    if EXECUTION_MODE == 'BUILD_SUBSTRATE':
        command += ['build-substrate', '--train-csv', str(TRAIN_CSV), '--dictionary-npz', str(DICTIONARY_NPZ), '--train-model-npz', str(TRAIN_MODEL_NPZ), '--output-dir', str(OUTPUT_DIR)]
    elif EXECUTION_MODE == 'RUN_REPLICATE':
        if STABILITY_ARM not in {'M', 'C'} or not STABILITY_REPLICATE_ID.isdigit() or not 0 <= int(STABILITY_REPLICATE_ID) < 20:
            raise ValueError('replicate execution requires arm M/C and replicate ID 0..19')
        command += ['run-replicate', '--arm', STABILITY_ARM, '--replicate-id', STABILITY_REPLICATE_ID, '--substrate-dir', str(SUBSTRATE_DIR), '--train-model-npz', str(TRAIN_MODEL_NPZ), '--output-dir', str(OUTPUT_DIR)]
    else:
        command += ['finalize-train', '--train-csv', str(TRAIN_CSV), '--substrate-dir', str(SUBSTRATE_DIR), '--replicates-dir', str(REPLICATES_DIR), '--output-dir', str(OUTPUT_DIR)]
    environment = os.environ.copy()
    environment['MOTIF_SCIENTIFIC_SHA'] = SOURCE_SHA
    subprocess.run(command, cwd=REPO_DIR, env=environment, check=True)
    inventory = {path.name: {'bytes': path.stat().st_size, 'sha256': sha256_file(path)} for path in sorted(OUTPUT_DIR.iterdir()) if path.is_file()}
    print(json.dumps({'status': 'COMPLETE', 'execution_mode': EXECUTION_MODE, 'outputs': inventory}, indent=2, sort_keys=True))
